# VinBigData teljes export 1024 JPG-re + bbox CSV transzformáció

Ez a notebook a **VinBigData Chest X-ray Abnormalities Detection** teljes train adathalmazán:

1. az összes train DICOM képet aránytartó módon 1024×1024-re alakítja,
2. az összes képet JPG formátumba menti,
3. az eredeti annotációs CSV összes bbox-os sorát ugyanebbe a 1024×1024 koordinátarendszerbe transzformálja,
4. újrafuttatásnál cache-eket használ, hogy gyorsabb legyen.

## Kimenet

A fő kimenetek:

- `jpg_1024/` – az összes exportált JPG kép
- `annotations_1024.csv` – az eredeti annotációk bbox-ainak 1024-es koordinátákra transzformált változata

## Geometria

A kimeneti kép mindig 1024×1024, de az eredeti képet nem torzítjuk:

- aránytartó resize,
- majd középre igazított padding.

A bbox-ok ugyanebben a transzformált koordinátarendszerben kerülnek mentésre.

In [14]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
from PIL import Image

# Konfiguráció
TARGET_SIZE = 1024
JPEG_QUALITY = 95

DEBUG_MAX_IMAGES = None
# DEBUG_MAX_IMAGES = 200

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/vinbig_full_jpg_1024")
JPG_DIR = OUTPUT_DIR / "jpg_1024"

ANNOTATION_CANDIDATE_NAMES = ["train.csv", "annotations_train.csv"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JPG_DIR.mkdir(parents=True, exist_ok=True)

# Rerun / cache
REUSE_EXISTING_PATH_INDEX = True
REUSE_EXISTING_MANIFEST = True
SAVE_PATH_INDEX_CACHE = True
SAVE_IMAGE_MANIFEST = True

print("OUTPUT_DIR =", OUTPUT_DIR)
print("JPG export =", JPG_DIR)

OUTPUT_DIR = /kaggle/working/vinbig_full_jpg_1024
JPG export = /kaggle/working/vinbig_full_jpg_1024/jpg_1024


In [15]:
# Segédfüggvények
def human_bytes(n_bytes):
    n = float(n_bytes)
    units = ["B", "KB", "MB", "GB", "TB"]
    for unit in units:
        if n < 1024 or unit == units[-1]:
            return f"{n:.2f} {unit}"
        n /= 1024

def find_annotation_and_train_dir(input_root: Path, candidate_names):
    annotation_candidates = []
    for name in candidate_names:
        annotation_candidates.extend(input_root.rglob(name))

    if not annotation_candidates:
        raise FileNotFoundError(
            f"Nem találtam {candidate_names} egyikét sem a {input_root} alatt."
        )

    ann_path = None
    for p in annotation_candidates:
        if (p.parent / "train").exists():
            ann_path = p
            break
    if ann_path is None:
        ann_path = annotation_candidates[0]

    preferred_train_dir = ann_path.parent / "train"
    if preferred_train_dir.exists():
        train_dir = preferred_train_dir
    else:
        train_dir_candidates = [p for p in input_root.rglob("*") if p.is_dir() and p.name.lower() == "train"]
        if not train_dir_candidates:
            raise FileNotFoundError("Nem találtam train könyvtárat a /kaggle/input alatt.")
        train_dir = train_dir_candidates[0]

    return ann_path, train_dir

def normalize_to_uint8(img: np.ndarray) -> np.ndarray:
    arr = img.astype(np.float32)

    lo, hi = np.percentile(arr, (0.5, 99.5))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(arr.min()), float(arr.max())

    if hi <= lo:
        return np.zeros(arr.shape, dtype=np.uint8)

    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / (hi - lo)
    arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
    return arr

def load_dicom_as_uint8(path: str) -> np.ndarray:
    ds = pydicom.dcmread(path, force=True)
    try:
        img = apply_voi_lut(ds.pixel_array, ds)
    except Exception:
        img = ds.pixel_array

    img = img.astype(np.float32)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    slope = getattr(ds, "RescaleSlope", None)
    intercept = getattr(ds, "RescaleIntercept", None)
    if slope is not None:
        img = img * float(slope)
    if intercept is not None:
        img = img + float(intercept)

    return normalize_to_uint8(img)

def compute_letterbox_meta(orig_h: int, orig_w: int, target_size: int = 1024):
    scale = min(target_size / orig_w, target_size / orig_h)
    new_w = max(1, int(round(orig_w * scale)))
    new_h = max(1, int(round(orig_h * scale)))
    pad_left = (target_size - new_w) // 2
    pad_top = (target_size - new_h) // 2

    return {
        "orig_w": int(orig_w),
        "orig_h": int(orig_h),
        "new_w": int(new_w),
        "new_h": int(new_h),
        "scale": float(scale),
        "pad_left": int(pad_left),
        "pad_top": int(pad_top),
        "target_size": int(target_size),
    }

def read_dicom_hw(path: str):
    ds = pydicom.dcmread(
        path,
        stop_before_pixels=True,
        specific_tags=["Rows", "Columns"],
        force=True,
    )
    return int(ds.Rows), int(ds.Columns)

def resize_with_letterbox(img_uint8: np.ndarray, target_size: int = 1024):
    h, w = img_uint8.shape[:2]
    meta = compute_letterbox_meta(h, w, target_size)

    pil_img = Image.fromarray(img_uint8)
    resized = pil_img.resize((meta["new_w"], meta["new_h"]), resample=Image.Resampling.BILINEAR)

    canvas = Image.new("L", (target_size, target_size), color=0)
    canvas.paste(resized, (meta["pad_left"], meta["pad_top"]))
    return canvas, meta

def transform_bbox(x_min, y_min, x_max, y_max, scale, pad_left, pad_top):
    return (
        x_min * scale + pad_left,
        y_min * scale + pad_top,
        x_max * scale + pad_left,
        y_max * scale + pad_top,
    )

def index_dicom_files(train_dir: Path):
    rows = []
    for p in tqdm(train_dir.rglob("*"), desc="Indexing DICOM files"):
        if p.is_file():
            suffix = p.suffix.lower()
            if suffix in {".dicom", ".dcm", ""}:
                rows.append({
                    "image_id": p.stem,
                    "dicom_path": str(p),
                    "file_size_bytes": p.stat().st_size,
                })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("Nem találtam DICOM fájlokat a train könyvtárban.")
    return df

def dir_size_bytes(folder: Path) -> int:
    total = 0
    if not folder.exists():
        return total
    for p in folder.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total

In [16]:
# Input felderítése
ann_path, train_dir = find_annotation_and_train_dir(INPUT_ROOT, ANNOTATION_CANDIDATE_NAMES)

print("Annotation file:", ann_path)
print("Train dir:", train_dir)

Annotation file: /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv
Train dir: /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train


In [17]:
# Annotációk betöltése
ann = pd.read_csv(ann_path)
ann.columns = [c.strip() for c in ann.columns]

required_cols = {"image_id", "class_name"}
missing = required_cols - set(ann.columns)
if missing:
    raise ValueError(f"Hiányzó kötelező oszlop(ok): {missing}")

for c in ["x_min", "y_min", "x_max", "y_max"]:
    if c not in ann.columns:
        ann[c] = np.nan

ann["has_bbox"] = (
    ann[["x_min", "y_min", "x_max", "y_max"]].notna().all(axis=1)
    & (ann["x_max"] > ann["x_min"])
    & (ann["y_max"] > ann["y_min"])
)

print("Annotációs sorok:", len(ann))
print("Egyedi képek az annotációkban:", ann["image_id"].nunique())
display(ann.head())

Annotációs sorok: 67914
Egyedi képek az annotációkban: 15000


,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max,has_bbox
0,50a418190bc3fb1ef1633bf9678929b3,No finding,14,R11,NaN,NaN,NaN,NaN,False
1,21a10246a5ec7af151081d0cd6d65dc9,No finding,14,R7,NaN,NaN,NaN,NaN,False
2,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,691.0,1375.0,1653.0,1831.0,True
3,051132a778e61a86eb147c7c6f564dfe,Aortic enlargement,0,R10,1264.0,743.0,1611.0,1019.0,True
4,063319de25ce7edb9b1c6b8881290140,No finding,14,R10,NaN,NaN,NaN,NaN,False


In [18]:
# DICOM path index betöltése vagy újraépítése
path_index_path = OUTPUT_DIR / "dicom_path_index.csv"

if REUSE_EXISTING_PATH_INDEX and path_index_path.exists():
    path_df = pd.read_csv(path_index_path)
    print("Korábbi DICOM path index betöltve:", path_index_path)
    print("Indexelt képek száma:", len(path_df))
else:
    path_df = index_dicom_files(train_dir)
    if SAVE_PATH_INDEX_CACHE:
        path_df.to_csv(path_index_path, index=False)
        print("Új DICOM path index mentve:", path_index_path)
    else:
        print("DICOM path index csak memóriában készült el.")
    print("Indexelt képek száma:", len(path_df))

manifest = path_df.copy()
manifest["jpg_path"] = manifest["image_id"].map(lambda x: str(JPG_DIR / f"{x}.jpg"))
manifest["file_exists"] = manifest["dicom_path"].notna()

if DEBUG_MAX_IMAGES is not None:
    manifest = manifest.head(DEBUG_MAX_IMAGES).copy()

print("Exportálandó képek száma:", manifest["image_id"].nunique())
display(manifest.head())

Korábbi DICOM path index betöltve: /kaggle/working/vinbig_full_jpg_1024/dicom_path_index.csv
Indexelt képek száma: 15000
Exportálandó képek száma: 15000


,image_id,dicom_path,file_size_bytes,jpg_path,file_exists
0,4d390e07733ba06e5ff07412f09c0a92,/kaggle/input/competitions/vinbigdata-chest-xr...,8582684,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,True
1,289f69f6462af4933308c275d07060f0,/kaggle/input/competitions/vinbigdata-chest-xr...,18874874,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,True
2,68335ee73e67706aa59b8b55b54b11a4,/kaggle/input/competitions/vinbigdata-chest-xr...,13250272,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,True
3,7ecd6f67f649f26c05805c8359f9e528,/kaggle/input/competitions/vinbigdata-chest-xr...,5827738,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,True
4,2229148faa205e881cf0d932755c9e40,/kaggle/input/competitions/vinbigdata-chest-xr...,6651622,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,True


In [19]:
# Korábbi manifest visszaolvasása újrafuttatás gyorsításához
image_manifest_path = OUTPUT_DIR / "image_manifest_1024.csv"

cached_manifest = None
cached_meta_by_id = None

if REUSE_EXISTING_MANIFEST and image_manifest_path.exists():
    cached_manifest = pd.read_csv(image_manifest_path)

    expected_cols = {
        "image_id", "dicom_path",
        "orig_w", "orig_h", "new_w", "new_h",
        "scale", "pad_left", "pad_top", "target_size",
        "jpg_path"
    }

    if expected_cols.issubset(set(cached_manifest.columns)):
        cached_manifest = cached_manifest.drop_duplicates(subset=["image_id"]).copy()
        cached_meta_by_id = cached_manifest.set_index("image_id")
        print("Korábbi manifest betöltve:", image_manifest_path)
        print("Cache-elt képek száma:", len(cached_meta_by_id))
    else:
        print("Van korábbi manifest, de hiányoznak belőle szükséges oszlopok, ezért nem használom cache-ként.")
else:
    print("Nem használok korábbi manifest cache-t.")

Korábbi manifest betöltve: /kaggle/working/vinbig_full_jpg_1024/image_manifest_1024.csv
Cache-elt képek száma: 14899


In [20]:
# Csak a hiányzó vagy cache nélküli képek feldolgozása
manifest = manifest.copy()
manifest["jpg_exists"] = manifest["jpg_path"].map(lambda p: Path(p).exists())

if cached_meta_by_id is not None:
    manifest["meta_cached"] = manifest["image_id"].isin(cached_meta_by_id.index)
else:
    manifest["meta_cached"] = False

manifest["fully_ready"] = manifest["jpg_exists"] & manifest["meta_cached"]

ready_manifest = manifest[manifest["fully_ready"]].copy()
to_process_manifest = manifest[~manifest["fully_ready"]].copy()

print("Már teljesen kész képek (JPG + meta cache):", len(ready_manifest))
print("További feldolgozást igénylő képek:", len(to_process_manifest))

if len(to_process_manifest) > 0:
    display(to_process_manifest[["image_id", "jpg_exists", "meta_cached"]].head())

Már teljesen kész képek (JPG + meta cache): 14899
További feldolgozást igénylő képek: 101


,image_id,jpg_exists,meta_cached
65,7d48ee923365d79f033147cd5f4aafc1,False,False
124,2fdd72337c9a875cf679564355b4bef2,False,False
280,d1dfd6ac02ac34ebaf9788ec9b2de1f3,False,False
467,6fbeb3ec1ec16b267f98fc12cbab9b6f,False,False
703,9074b5859c254e4cc0d1ae9781cfcdf1,False,False


In [21]:
# JPG export + meta összegyűjtése
transform_rows = []
error_rows = []

reused_from_manifest = 0
reused_header_only = 0
newly_exported = 0

if cached_meta_by_id is not None and len(ready_manifest) > 0:
    for row in ready_manifest.itertuples(index=False):
        meta_row = cached_meta_by_id.loc[row.image_id]
        transform_rows.append({
            "image_id": row.image_id,
            "dicom_path": row.dicom_path,
            "orig_w": int(meta_row["orig_w"]),
            "orig_h": int(meta_row["orig_h"]),
            "new_w": int(meta_row["new_w"]),
            "new_h": int(meta_row["new_h"]),
            "scale": float(meta_row["scale"]),
            "pad_left": int(meta_row["pad_left"]),
            "pad_top": int(meta_row["pad_top"]),
            "target_size": int(meta_row["target_size"]),
            "jpg_path": row.jpg_path,
            "export_status": "reused_from_manifest",
        })
        reused_from_manifest += 1

for row in tqdm(to_process_manifest.itertuples(index=False), total=len(to_process_manifest), desc="Exporting missing JPG images"):
    image_id = row.image_id
    dicom_path = row.dicom_path
    jpg_path = row.jpg_path
    jpg_exists = row.jpg_exists

    try:
        if jpg_exists:
            orig_h, orig_w = read_dicom_hw(dicom_path)
            meta = compute_letterbox_meta(orig_h, orig_w, TARGET_SIZE)

            transform_rows.append({
                "image_id": image_id,
                "dicom_path": dicom_path,
                "orig_w": meta["orig_w"],
                "orig_h": meta["orig_h"],
                "new_w": meta["new_w"],
                "new_h": meta["new_h"],
                "scale": meta["scale"],
                "pad_left": meta["pad_left"],
                "pad_top": meta["pad_top"],
                "target_size": meta["target_size"],
                "jpg_path": jpg_path,
                "export_status": "reused_header_only",
            })
            reused_header_only += 1
            continue

        img8 = load_dicom_as_uint8(dicom_path)
        canvas, meta = resize_with_letterbox(img8, TARGET_SIZE)
        canvas.save(jpg_path, format="JPEG", quality=JPEG_QUALITY, optimize=True)

        transform_rows.append({
            "image_id": image_id,
            "dicom_path": dicom_path,
            "orig_w": meta["orig_w"],
            "orig_h": meta["orig_h"],
            "new_w": meta["new_w"],
            "new_h": meta["new_h"],
            "scale": meta["scale"],
            "pad_left": meta["pad_left"],
            "pad_top": meta["pad_top"],
            "target_size": meta["target_size"],
            "jpg_path": jpg_path,
            "export_status": "newly_exported",
        })
        newly_exported += 1

    except Exception as e:
        error_rows.append({
            "image_id": image_id,
            "dicom_path": dicom_path,
            "error": repr(e),
        })

transform_df = pd.DataFrame(transform_rows)
error_df = pd.DataFrame(error_rows)

print("Manifestből újrahasznált képek:", reused_from_manifest)
print("Headerből újrahasznált képek:", reused_header_only)
print("Újonnan exportált képek:", newly_exported)
print("Hibás export:", len(error_df))
display(transform_df.head())
if not error_df.empty:
    display(error_df.head())

Exporting missing JPG images:   0%|          | 0/101 [00:00<?, ?it/s]

Manifestből újrahasznált képek: 14899
Headerből újrahasznált képek: 0
Újonnan exportált képek: 101
Hibás export: 0


,image_id,dicom_path,orig_w,orig_h,new_w,new_h,scale,pad_left,pad_top,target_size,jpg_path,export_status
0,4d390e07733ba06e5ff07412f09c0a92,/kaggle/input/competitions/vinbigdata-chest-xr...,3000,3000,1024,1024,0.341333,0,0,1024,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,reused_from_manifest
1,289f69f6462af4933308c275d07060f0,/kaggle/input/competitions/vinbigdata-chest-xr...,3072,3072,1024,1024,0.333333,0,0,1024,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,reused_from_manifest
2,68335ee73e67706aa59b8b55b54b11a4,/kaggle/input/competitions/vinbigdata-chest-xr...,2336,2836,843,1024,0.361072,90,0,1024,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,reused_from_manifest
3,7ecd6f67f649f26c05805c8359f9e528,/kaggle/input/competitions/vinbigdata-chest-xr...,2744,2952,952,1024,0.346883,36,0,1024,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,reused_from_manifest
4,2229148faa205e881cf0d932755c9e40,/kaggle/input/competitions/vinbigdata-chest-xr...,2304,2880,819,1024,0.355556,102,0,1024,/kaggle/working/vinbig_full_jpg_1024/jpg_1024/...,reused_from_manifest


In [22]:
# Teljes bbox-os annotációs CSV transzformációja
# Az összes bbox-os sor marad, nincs finding-szűrés.
bbox_ann = ann[ann["has_bbox"]].copy()

bbox_ann_1024 = bbox_ann.merge(
    transform_df[["image_id", "scale", "pad_left", "pad_top", "target_size"]],
    on="image_id",
    how="inner"
).copy()

tx = []
for row in bbox_ann_1024.itertuples(index=False):
    x1, y1, x2, y2 = transform_bbox(
        row.x_min, row.y_min, row.x_max, row.y_max,
        row.scale, row.pad_left, row.pad_top
    )
    tx.append((x1, y1, x2, y2))

tx = np.array(tx, dtype=np.float32)
bbox_ann_1024["x_min_1024"] = tx[:, 0]
bbox_ann_1024["y_min_1024"] = tx[:, 1]
bbox_ann_1024["x_max_1024"] = tx[:, 2]
bbox_ann_1024["y_max_1024"] = tx[:, 3]

for c in ["x_min_1024", "y_min_1024", "x_max_1024", "y_max_1024"]:
    bbox_ann_1024[c] = bbox_ann_1024[c].clip(0, TARGET_SIZE)

bbox_ann_1024["bbox_w_1024"] = bbox_ann_1024["x_max_1024"] - bbox_ann_1024["x_min_1024"]
bbox_ann_1024["bbox_h_1024"] = bbox_ann_1024["y_max_1024"] - bbox_ann_1024["y_min_1024"]

bbox_ann_1024 = bbox_ann_1024[
    (bbox_ann_1024["bbox_w_1024"] > 0) &
    (bbox_ann_1024["bbox_h_1024"] > 0)
].copy()

print("Transzformált bbox sorok:", len(bbox_ann_1024))
display(bbox_ann_1024.head())

Transzformált bbox sorok: 36096


,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max,has_bbox,scale,pad_left,pad_top,target_size,x_min_1024,y_min_1024,x_max_1024,y_max_1024,bbox_w_1024,bbox_h_1024
0,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,691.0,1375.0,1653.0,1831.0,True,0.438356,56,0,1024,358.904114,602.739746,780.602722,802.630127,421.698608,199.890381
1,051132a778e61a86eb147c7c6f564dfe,Aortic enlargement,0,R10,1264.0,743.0,1611.0,1019.0,True,0.355556,102,0,1024,551.422241,264.177765,674.799988,362.311096,123.377747,98.133331
2,1c32170b4af4ce1a3030eb8167753b06,Pleural thickening,11,R9,627.0,357.0,947.0,433.0,True,0.333333,88,0,1024,297.000000,119.000000,403.666656,144.333328,106.666656,25.333328
3,0c7a38f293d5f5e4846aa4ca6db4daf1,ILD,5,R17,1347.0,245.0,2188.0,2169.0,True,0.400783,54,0,1024,593.854431,98.191780,930.912720,869.297852,337.058289,771.106079
4,47ed17dcb2cbeec15182ed335a8b5a9e,Nodule/Mass,8,R9,557.0,2352.0,675.0,2484.0,True,0.305398,120,0,1024,290.106781,718.296448,326.143738,758.609009,36.036957,40.312561


In [23]:
# Kimenetek mentése
# Fő kimenet: JPG mappa + egy bbox CSV
# Cache: path index + image manifest
image_manifest_path = OUTPUT_DIR / "image_manifest_1024.csv"
annotations_1024_path = OUTPUT_DIR / "annotations_1024.csv"

final_manifest = manifest.drop(columns=["jpg_exists", "meta_cached", "fully_ready"], errors="ignore").merge(
    transform_df,
    on=["image_id", "dicom_path", "jpg_path"],
    how="left"
).copy()

if SAVE_IMAGE_MANIFEST:
    final_manifest.to_csv(image_manifest_path, index=False)

bbox_ann_1024.to_csv(annotations_1024_path, index=False)

print("Saved:")
print(" -", annotations_1024_path)
if SAVE_IMAGE_MANIFEST:
    print(" -", image_manifest_path, "(cache)")
if SAVE_PATH_INDEX_CACHE:
    print(" -", OUTPUT_DIR / "dicom_path_index.csv", "(cache)")

Saved:
 - /kaggle/working/vinbig_full_jpg_1024/annotations_1024.csv
 - /kaggle/working/vinbig_full_jpg_1024/image_manifest_1024.csv (cache)
 - /kaggle/working/vinbig_full_jpg_1024/dicom_path_index.csv (cache)


In [24]:
# Rövid összefoglaló
jpg_size = dir_size_bytes(JPG_DIR)

print("Összes exportált / ismert kép:", len(transform_df))
print("JPG összméret:", human_bytes(jpg_size))
print("BBox-os annotációs sorok 1024-re transzformálva:", len(bbox_ann_1024))
print("Hibás export:", len(error_df))

Összes exportált / ismert kép: 15000
JPG összméret: 2.64 GB
BBox-os annotációs sorok 1024-re transzformálva: 36096
Hibás export: 0


In [25]:
from pathlib import Path
import shutil

jpg_zip_path = shutil.make_archive(
    str(OUTPUT_DIR / "jpg_1024"),
    "zip",
    root_dir=str(JPG_DIR)
)

print("Létrehozott zip:")
print(jpg_zip_path)

Létrehozott zip:
/kaggle/working/vinbig_full_jpg_1024/jpg_1024.zip
